# Practice Lab: Bias Detection in Code (Lesson 15.1)

Module 15 . 3 exercises . 5-demographic test + job rec bias + CI/CD bias gate


# Lesson 15.1: Bias Detection in Code

3 exercises: 1E + 1M + 1C

Module 15: Ethics, Safety, Governance & Explainability


In [ ]:
import random, numpy as np
from collections import Counter
from datetime import datetime
random.seed(42); np.random.seed(42)


---
## Exercise 1 (Easy): 5-Demographic Test


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import random, numpy as np
from collections import Counter
random.seed(42); np.random.seed(42)

class DBT:
    def __init__(self):
        self.names={"Priya Sharma":{"w":[0.5,0.35,0.15],"lm":95,"ls":12},
            "James Smith":{"w":[0.7,0.25,0.05],"lm":120,"ls":15},
            "Wei Zhang":{"w":[0.55,0.35,0.10],"lm":105,"ls":14},
            "Fatima Hassan":{"w":[0.45,0.35,0.20],"lm":90,"ls":11},
            "Oluwaseun Adeyemi":{"w":[0.50,0.30,0.20],"lm":88,"ls":13}}
        self.sents=["positive","neutral","negative"]
    def sim(self,n=10):
        r={}
        for nm,p in self.names.items():
            s=random.choices(self.sents,weights=p["w"],k=n)
            l=[max(20,round(random.gauss(p["lm"],p["ls"]))) for _ in range(n)]
            r[nm]={"s":s,"l":l}
        return r
    def chi2(self,r):
        ct=np.array([[r[n]["s"].count(s) for s in self.sents] for n in self.names])
        rs=ct.sum(1,keepdims=True); cs=ct.sum(0,keepdims=True); t=ct.sum()
        ex=np.where(rs*cs/t==0,1,rs*cs/t)
        chi=((ct-ex)**2/ex).sum(); dof=(len(self.names)-1)*2
        crit={4:9.49,6:12.59,8:15.51}.get(dof,15.51)
        return {"chi2":round(chi,2),"sig":chi>crit}
    def boot(self,a,b,nb=2000):
        obs=abs(np.mean(a)-np.mean(b)); c=a+b; cnt=0
        for _ in range(nb):
            np.random.shuffle(c); d=abs(np.mean(c[:len(a)])-np.mean(c[len(a):]))
            if d>=obs: cnt+=1
        return {"diff":round(obs,1),"p":round(cnt/nb,4),"sig":cnt/nb<0.05}

t=DBT(); r=t.sim(10)
print("5-Demographic Bias Test:")
print(f"  {'Name':<22} {'Pos':>4} {'Neu':>4} {'Neg':>4} {'AvgLen':>7}")
for nm,d in r.items():
    sc=Counter(d["s"]); al=round(np.mean(d["l"]),1)
    print(f"  {nm:<22} {sc['positive']:>4} {sc['neutral']:>4} {sc['negative']:>4} {al:>7}")
c=t.chi2(r)
print(f"\n  Chi2={c['chi2']} Bias={'YES' if c['sig'] else 'NO'}")
nms=list(t.names.keys()); bp=[]
for i in range(len(nms)):
    for j in range(i+1,len(nms)):
        b=t.boot(r[nms[i]]["l"][:],r[nms[j]]["l"][:])
        if b["sig"]: bp.append(f"    {nms[i]} vs {nms[j]}: diff={b['diff']} p={b['p']}")
print(f"\n  Length bias pairs: {len(bp)}")
for p in bp: print(p)


</details>


---
## Exercise 2 (Medium): Job Recommendation Bias


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import random, numpy as np
from collections import Counter
random.seed(42)

NAMES={"Priya Sharma":("technical",0.1),"James Smith":("technical",0.3),"Wei Zhang":("technical",0.25),
    "Fatima Hassan":("service",0.2),"Maria Garcia":("creative",0.15)}
FIELDS={"CS":[0.70,0.15,0.15],"Literature":[0.15,0.65,0.20],"Biology":[0.40,0.20,0.40]}
TYPES=["technical","creative","service"]

def sim(n=5):
    r={}
    for nm,(bt,bs) in NAMES.items():
        for fld,bd in FIELDS.items():
            w=list(bd); bi=TYPES.index(bt)
            for i in range(3): w[i]+=(bs if i==bi else -bs/2)
            w=[max(0.05,x) for x in w]; s=sum(w); w=[x/s for x in w]
            r[f"{nm}|{fld}"]=random.choices(TYPES,weights=w,k=n)
    return r

def chi2_field(r,field):
    ct=np.array([[r[f"{nm}|{field}"].count(t) for t in TYPES] for nm in NAMES])
    rs=ct.sum(1,keepdims=True); cs=ct.sum(0,keepdims=True); tot=ct.sum()
    ex=np.where(rs*cs/tot==0,1,rs*cs/tot)
    chi=((ct-ex)**2/ex).sum(); dof=(len(NAMES)-1)*2
    return {"chi2":round(chi,2),"sig":chi>{4:9.49,6:12.59,8:15.51}.get(dof,15.51)}

r=sim(5)
print("Job Recommendation Bias:")
for fld in FIELDS:
    print(f"\n  {fld}:")
    print(f"  {'Name':<22} {'Tech':>5} {'Creat':>6} {'Serv':>5}")
    for nm in NAMES:
        tc=Counter(r[f"{nm}|{fld}"])
        print(f"  {nm:<22} {tc['technical']:>5} {tc['creative']:>6} {tc['service']:>5}")
    c=chi2_field(r,fld)
    print(f"  Chi2={c['chi2']} Bias={'YES' if c['sig'] else 'NO'}")

biased=[f for f in FIELDS if chi2_field(r,f)["sig"]]
print(f"\n  Biased fields: {biased if biased else 'None'}")


</details>


---
## Exercise 3 (Challenge): CI/CD Bias Gate


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import random, numpy as np
from datetime import datetime
random.seed(42); np.random.seed(42)

class BG:
    def __init__(self,alpha=0.05):
        self.alpha=alpha; self.tested=0; self.biases=[]; self.results=[]
    def sim_test(self,tmpl,demos,n=10):
        bias={"job_rec":["Fatima Hassan","Oluwaseun Adeyemi"],"loan":["Priya Sharma","Maria Garcia"],
            "review":[],"medical":[],"admission":["Wei Zhang"]}
        affected=bias.get(tmpl,[])
        r={}
        for nm in demos:
            if nm in affected: w=[0.3,0.4,0.3]; lm=75
            else: w=[0.65,0.30,0.05]; lm=110
            r[nm]={"s":random.choices(["positive","neutral","negative"],weights=w,k=n),
                   "l":[max(20,round(random.gauss(lm,12))) for _ in range(n)]}
        return r
    def chi2(self,r,demos):
        ct=np.array([[r[n]["s"].count(s) for s in ["positive","neutral","negative"]] for n in demos])
        rs=ct.sum(1,keepdims=True); cs=ct.sum(0,keepdims=True); t=ct.sum()
        ex=np.where(rs*cs/t==0,1,rs*cs/t)
        chi=((ct-ex)**2/ex).sum(); dof=(len(demos)-1)*2
        return {"chi2":round(chi,2),"sig":chi>{4:9.49,6:12.59,8:15.51}.get(dof,15.51)}
    def boot_max(self,r,demos,nb=1000):
        mx=0; mp=""
        for i in range(len(demos)):
            for j in range(i+1,len(demos)):
                d=abs(np.mean(r[demos[i]]["l"])-np.mean(r[demos[j]]["l"]))
                if d>mx: mx=d; mp=f"{demos[i]} vs {demos[j]}"
        if not mp: return {"sig":False}
        pts=mp.split(" vs "); a=r[pts[0]]["l"][:]; b=r[pts[1]]["l"][:]
        obs=abs(np.mean(a)-np.mean(b)); c=a+b; cnt=0
        for _ in range(nb):
            np.random.shuffle(c); dd=abs(np.mean(c[:len(a)])-np.mean(c[len(a):]))
            if dd>=obs: cnt+=1
        p=cnt/nb
        return {"pair":mp,"diff":round(mx,1),"p":round(p,4),"sig":p<self.alpha}
    def run(self,tmpls,demos,n=10):
        for tmpl in tmpls:
            self.tested+=1; r=self.sim_test(tmpl,demos,n)
            c=self.chi2(r,demos); b=self.boot_max(r,demos)
            biased=c["sig"] or b.get("sig",False)
            self.results.append({"t":tmpl,"chi2":c["chi2"],"sb":c["sig"],"lb":b.get("sig",False)})
            if biased: self.biases.append({"t":tmpl,"sc":c["sig"],"lc":b.get("sig",False),"chi2":c["chi2"]})
        return len(self.biases)==0

g=BG()
tmpls=["job_rec","loan","review","medical","admission"]
demos=["Priya Sharma","James Smith","Wei Zhang","Fatima Hassan","Oluwaseun Adeyemi"]

passed=g.run(tmpls,demos,10)
print("CI/CD Bias Gate Report:")
print(f"  Templates: {g.tested} | Biases: {len(g.biases)} | Status: {'PASS' if passed else 'BLOCKED'}")
print(f"\n  {'Template':<14} {'Chi2':>6} {'Sent?':>6} {'Len?':>6} {'Result':>8}")
for r in g.results:
    res="FAIL" if r["sb"] or r["lb"] else "PASS"
    print(f"  {r['t']:<14} {r['chi2']:>6.1f} {'YES' if r['sb'] else 'no':>6} {'YES' if r['lb'] else 'no':>6} {res:>8}")
if g.biases:
    print(f"\n  Biased templates: {[b['t'] for b in g.biases]}")
print(f"\n  Exit code: {0 if passed else 1} ({'deploy' if passed else 'BLOCKED'})")


</details>
